# 문서 이미지 자동 교정 및 향상 시스템
# 文档图像自动矫正与增强系统
# Document Image Auto-Correction & Enhancement

**Google Colab Edition**

스마트폰으로 촬영된 문서 이미지의 기하학적 왜곡을 보정하고 텍스트 가독성을 향상시키는 시스템입니다.
本系统用于矫正手机拍摄文档图像的几何失真并提高文本可读性。

- ⚠️ Machine Learning / Deep Learning 사용 금지 / 禁止使用机器学习/深度学习
- ✅ 전통적 Computer Vision + Image Processing 기법만 사용 / 仅使用传统计算机视觉与图像处理技术 (OpenCV, NumPy)

## Pipeline (11 Steps / 11단계 / 11步骤)

```
Input → Grayscale → Gaussian Blur → Canny Edge → Edge Dilation
      → Contour Detection → Convex Hull → Quad Approximation
      → Corner Ordering → Perspective Transform
      → CLAHE → Adaptive Threshold → Morphology → Output
```

In [ ]:
# 1. 환경 설정 및 라이브러리 import
# 1. 环境设置与库导入
# Google Colab에는 OpenCV, NumPy, Matplotlib가 기본 설치되어 있습니다.
# Google Colab 已预装 OpenCV、NumPy、Matplotlib。
# 만약 누락된 패키지가 있다면 아래 주석을 해제하세요 / 如有缺失请取消下面注释:
# !pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import os
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files
from google.colab.patches import cv2_imshow
from IPython.display import display, Image as IPImage, HTML
import ipywidgets as widgets
from IPython.display import clear_output

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print("All libraries loaded successfully! / 모든 라이브러리 로드 완료! / 所有库加载成功！")

---
## 2. Utility Functions / 유틸리티 함수 / 工具函数 (Image I/O)

In [ ]:
# 2. 이미지 입출력 유틸리티 / 图像输入输出工具

def load_image(path):
    """Load image from file path as BGR ndarray.
    이미지 파일을 BGR ndarray로 로드합니다.
    从文件路径加载图像为BGR ndarray。
    Uses: cv2.imread()"""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Image file not found / 파일 없음 / 文件不存在: {path}")
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Failed to read image / 이미지 읽기 실패 / 图像读取失败: {path}")
    return img


def save_image(img, path):
    """Save image to disk.
    이미지를 디스크에 저장합니다.
    将图像保存到磁盘。
    Uses: cv2.imwrite()"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    cv2.imwrite(path, img)


def save_step(img, output_dir, step_num, step_name):
    """Save intermediate result as '{step_num:02d}_{step_name}.jpg'.
    중간 결과를 '{step_num:02d}_{step_name}.jpg' 형식으로 저장합니다.
    将中间结果保存为 '{step_num:02d}_{step_name}.jpg' 格式。
    Uses: cv2.imwrite()"""
    filename = f"{step_num:02d}_{step_name}.jpg"
    filepath = os.path.join(output_dir, filename)
    os.makedirs(output_dir, exist_ok=True)
    cv2.imwrite(filepath, img)


def show_images(images, titles, cols=3, figsize=(18, 10)):
    """Display multiple images in a grid using matplotlib.
    여러 이미지를 matplotlib 그리드로 표시합니다.
    使用matplotlib以网格形式显示多张图像。"""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:
            axes[i].imshow(img, cmap='gray')
        else:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

print("Utility functions defined. / 유틸리티 함수 정의 완료. / 工具函数已定义。")

---
## 3. Preprocessing Module / 전처리 모듈 / 预处理模块
Step 2-3: Grayscale Conversion + Gaussian Filtering

In [ ]:
# 3. 전처리 모듈 / 预处理模块

def to_grayscale(img):
    """Convert BGR to grayscale.
    BGR 이미지를 그레이스케일로 변환합니다.
    将BGR图像转换为灰度图。
    Formula / 공식 / 公式: Gray = 0.299R + 0.587G + 0.114B
    Uses: cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)"""
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def apply_gaussian_blur(gray, kernel_size=(5, 5), sigma=0):
    """Apply Gaussian filter to suppress high-frequency noise.
    가우시안 필터를 적용하여 고주파 노이즈를 제거합니다.
    应用高斯滤波以抑制高频噪声。
    Uses: cv2.GaussianBlur(gray, kernel_size, sigma)"""
    return cv2.GaussianBlur(gray, kernel_size, sigma)

print("Preprocessing functions defined. / 전처리 함수 정의 완료. / 预处理函数已定义。")

---
## 4. Detection Module / 문서 검출 모듈 / 文档检测模块
Step 4-6: Canny Edge → Edge Dilation → Contour Detection → Quadrilateral Approximation

In [ ]:
# 4. 문서 검출 모듈 / 文档检测模块

def detect_edges(blurred, low=75, high=200):
    """Extract edges from blurred grayscale image.
    블러 처리된 그레이스케일 이미지에서 에지를 추출합니다.
    从模糊灰度图像中提取边缘。
    Uses: cv2.Canny(blurred, low, high)"""
    return cv2.Canny(blurred, low, high)


def dilate_edges(edges, kernel_size=(3, 3), iterations=2):
    """Dilate edge map to connect fragmented edge segments.
    에지 맵을 팽창하여 조각난 에지 세그먼트를 연결합니다.
    膨胀边缘图以连接碎片化的边缘段。
    Uses: cv2.dilate()"""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    return cv2.dilate(edges, kernel, iterations=iterations)


def find_contours(edges):
    """Extract contours sorted by area descending (top 10).
    면적 기준 내림차순으로 정렬된 윤곽선을 추출합니다 (상위 10개).
    提取轮廓并按面积降序排序（前10个）。
    Uses: cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)"""
    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    return contours[:10]


def _approx_quad(contour):
    """Try polygon approximation at various epsilon ratios.
    다양한 epsilon 비율로 다각형 근사를 시도합니다.
    以不同的epsilon比例尝试多边形逼近。
    Uses: cv2.approxPolyDP()"""
    peri = cv2.arcLength(contour, True)
    for ratio in (0.02, 0.03, 0.05, 0.08, 0.10):
        epsilon = ratio * peri
        approx = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) == 4:
            return approx
    return None


def find_document_contour(contours, img_area=0):
    """Find the best quadrilateral document candidate.
    최적의 사각형 문서 후보를 찾습니다.
    寻找最佳四边形文档候选区域。
    Strategy / 전략 / 策略:
      1. Direct polygon approximation per contour / 각 윤곽선 직접 다각형 근사 / 对每个轮廓直接进行多边形逼近
      2. Convex hull + approximation for partial edges / 부분 에지에 대한 볼록 껍질 + 근사 / 凸包+逼近处理部分边缘
      3. Combined contour convex hull as fallback / 전체 윤곽선 결합 볼록 껍질 폴백 / 组合所有轮廓的凸包作为后备方案
    Uses: cv2.approxPolyDP(), cv2.convexHull(), cv2.contourArea()
    """
    min_area = img_area * 0.02 if img_area > 0 else 5000

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        quad = _approx_quad(contour)
        if quad is not None:
            return quad
        hull = cv2.convexHull(contour)
        quad = _approx_quad(hull)
        if quad is not None:
            return quad

    if len(contours) >= 2:
        combined = np.vstack(contours)
        hull = cv2.convexHull(combined)
        hull_area = cv2.contourArea(hull)
        if hull_area >= min_area:
            quad = _approx_quad(hull)
            if quad is not None:
                return quad
    return None


def draw_contour(img, contour):
    """Draw the detected document boundary in green.
    검출된 문서 경계를 녹색으로 그립니다.
    以绿色绘制检测到的文档边界。
    Uses: cv2.drawContours()"""
    result = img.copy()
    cv2.drawContours(result, [contour], -1, (0, 255, 0), 3)
    return result

print("Detection functions defined. / 검출 함수 정의 완료. / 检测函数已定义。")

---
## 5. Geometry Module / 기하학적 보정 모듈 / 几何校正模块
Step 7-8: Corner Ordering + Perspective Transformation

In [ ]:
# 5. 기하학적 보정 모듈 / 几何校正模块

def _dist(p1, p2):
    """Euclidean distance between two points.
    두 점 사이의 유클리드 거리.
    两点之间的欧氏距离。"""
    return np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)


def order_points(pts):
    """Order 4 corner points as [TL, TR, BR, BL].
    4개 꼭짓점을 [TL, TR, BR, BL] 순서로 정렬합니다.
    将4个角点排序为 [左上, 右上, 右下, 左下]。
    TL: min x+y,  BR: max x+y,  TR: max x-y,  BL: min x-y.
    Returns (4,2) float32 ndarray."""
    pts = pts.reshape(4, 2)
    ordered = np.zeros((4, 2), dtype=np.float32)
    sums = pts.sum(axis=1)
    diffs = np.diff(pts, axis=1)
    ordered[0] = pts[np.argmin(sums)]   # TL / 좌상단 / 左上
    ordered[2] = pts[np.argmax(sums)]   # BR / 우하단 / 右下
    ordered[1] = pts[np.argmax(diffs)]  # TR / 우상단 / 右上
    ordered[3] = pts[np.argmin(diffs)]  # BL / 좌하단 / 左下
    return ordered


def calculate_output_size(ordered_pts):
    """Compute output width and height from ordered corner points.
    정렬된 꼭짓점으로부터 출력 너비와 높이를 계산합니다.
    从排序后的角点计算输出宽度和高度。
    width  = max(dist(BL,BR), dist(TL,TR))
    height = max(dist(TR,BR), dist(TL,BL))"""
    tl, tr, br, bl = ordered_pts
    width = int(max(_dist(bl, br), _dist(tl, tr)))
    height = int(max(_dist(tr, br), _dist(tl, bl)))
    return width, height


def apply_perspective_transform(img, ordered_pts):
    """Apply perspective correction to straighten the document.
    투시 변환을 적용하여 문서를 반듯하게 보정합니다.
    应用透视变换将文档拉直。
    Uses: cv2.getPerspectiveTransform(), cv2.warpPerspective()"""
    width, height = calculate_output_size(ordered_pts)
    dst_pts = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ], dtype=np.float32)
    M = cv2.getPerspectiveTransform(ordered_pts, dst_pts)
    warped = cv2.warpPerspective(img, M, (width, height))
    return warped

print("Geometry functions defined. / 기하 함수 정의 완료. / 几何函数已定义。")

---
## 6. Enhancement Module / 이미지 향상 모듈 / 图像增强模块
Step 9-10: CLAHE + Adaptive Thresholding + Morphological Processing

In [ ]:
# 6. 이미지 향상 모듈 / 图像增强模块

def apply_clahe(img_gray, clip_limit=2.0, tile_size=(8, 8)):
    """Enhance local contrast using CLAHE.
    CLAHE를 사용하여 지역 대비를 향상시킵니다.
    使用CLAHE增强局部对比度。
    Uses: cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)"""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    return clahe.apply(img_gray)


def apply_adaptive_threshold(img_gray, block_size=11, C=2):
    """Binarize image with adaptive thresholding to handle uneven illumination.
    적응형 임계값 처리로 불균일한 조명을 보정하며 이미지를 이진화합니다.
    使用自适应阈值处理来应对不均匀光照，对图像进行二值化。
    Uses: cv2.adaptiveThreshold()"""
    return cv2.adaptiveThreshold(
        img_gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size, C,
    )


def apply_morphology(img_binary, kernel_size=(3, 3)):
    """Clean binary image with morphological closing.
    형태학적 닫힘 연산으로 이진 이미지의 잡음을 정리합니다.
    使用形态学闭运算清理二值图像噪声。
    Uses: cv2.morphologyEx(MORPH_CLOSE)"""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    return cv2.morphologyEx(img_binary, cv2.MORPH_CLOSE, kernel)


def enhance(warped):
    """Run full enhancement pipeline on the perspective-corrected image.
    투시 보정된 이미지에 전체 향상 파이프라인을 적용합니다.
    对透视矫正后的图像执行完整增强流水线。
    Steps: grayscale → CLAHE → adaptive threshold → morphology.
    Returns BGR image."""
    gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    clahed = apply_clahe(gray)
    thresh = apply_adaptive_threshold(clahed)
    cleaned = apply_morphology(thresh)
    return cv2.cvtColor(cleaned, cv2.COLOR_GRAY2BGR)

print("Enhancement functions defined. / 향상 함수 정의 완료. / 增强函数已定义。")

---
## 7. Full Pipeline / 전체 파이프라인 / 完整流水线

In [ ]:
# 7. 전체 파이프라인 / 完整流水线

def run_pipeline(image, output_dir="output", save_steps=True, basename="document"):
    """Run the full 11-step document correction pipeline.
    전체 11단계 문서 교정 파이프라인을 실행합니다.
    运行完整的11步文档矫正流水线。

    Args / 인자 / 参数:
        image: BGR image (numpy array) or path string.
               BGR 이미지(numpy 배열) 또는 파일 경로 문자열.
               BGR图像(numpy数组)或文件路径字符串。
        output_dir: Directory for output images / 출력 디렉터리 / 输出目录.
        save_steps: Save intermediate step images / 중간 단계 이미지 저장 여부 / 是否保存中间步骤图像.
        basename: Base name for output files / 출력 파일 기본 이름 / 输出文件基础名称.

    Returns / 반환값 / 返回值:
        dict: {success, output_path, processing_time, steps_images, warped, result}
    """
    start_time = time.time()
    output_path = os.path.join(output_dir, f"{basename}_corrected.jpg")
    steps_dir = os.path.join(output_dir, f"{basename}_steps") if save_steps else None
    steps_images = {}

    # Load image if path provided / 경로면 로드, 배열이면 복사 / 如果是路径则加载，否则复制
    if isinstance(image, str):
        original = load_image(image)
    else:
        original = image.copy()

    if save_steps:
        save_step(original, steps_dir, 1, "original")
        steps_images['01_original'] = original.copy()

    # Step 2: Grayscale / 그레이스케일 / 灰度化
    gray = to_grayscale(original)
    if save_steps:
        save_step(gray, steps_dir, 2, "grayscale")
        steps_images['02_grayscale'] = gray.copy()

    # Step 3: Gaussian blur / 가우시안 블러 / 高斯模糊
    blurred = apply_gaussian_blur(gray)
    if save_steps:
        save_step(blurred, steps_dir, 3, "blurred")
        steps_images['03_blurred'] = blurred.copy()

    # Step 4: Canny edge detection / 케니 에지 검출 / Canny边缘检测
    edges = detect_edges(blurred)
    if save_steps:
        save_step(edges, steps_dir, 4, "edges")
        steps_images['04_edges'] = edges.copy()

    # Step 4b: Dilate edges to connect fragments / 에지 팽창으로 조각 연결 / 膨胀边缘连接碎片
    dilated = dilate_edges(edges)

    # Step 5-6: Contour detection + quadrilateral approximation / 윤곽선 검출 + 사각형 근사 / 轮廓检测+四边形逼近
    contours = find_contours(dilated)
    img_area = original.shape[0] * original.shape[1]
    doc_contour = find_document_contour(contours, img_area)

    if save_steps and contours:
        display_contour = doc_contour if doc_contour is not None else contours[0]
        contour_img = draw_contour(original, display_contour)
        save_step(contour_img, steps_dir, 5, "contours")
        steps_images['05_contours'] = contour_img.copy()

    if doc_contour is not None:
        # Step 7-8: Corner ordering + perspective transform / 꼭짓점 정렬 + 투시 변환 / 角点排序+透视变换
        ordered_pts = order_points(doc_contour)
        warped = apply_perspective_transform(original, ordered_pts)
        if save_steps:
            save_step(warped, steps_dir, 6, "perspective")
            steps_images['06_perspective'] = warped.copy()

        # Step 9-10: Enhancement / 화질 향상 / 图像增强
        result = enhance(warped)
        if save_steps:
            save_step(result, steps_dir, 7, "enhanced")
            steps_images['07_enhanced'] = result.copy()

        success = True
    else:
        # Detection failed — fallback to enhancement on original
        # 검출 실패 — 원본에 향상 처리만 적용 / 检测失败 — 仅对原图进行增强
        result = enhance(original)
        if save_steps:
            save_step(original, steps_dir, 6, "perspective")
            save_step(result, steps_dir, 7, "enhanced")
            steps_images['06_perspective'] = original.copy()
            steps_images['07_enhanced'] = result.copy()
        warped = original
        success = False

    save_image(result, output_path)
    elapsed = time.time() - start_time

    return {
        "success": success,
        "output_path": output_path,
        "processing_time": elapsed,
        "steps_images": steps_images,
        "warped": warped,
        "result": result,
        "original": original,
    }

print("Pipeline function defined. / 파이프라인 함수 정의 완료. / 流水线函数已定义。")

---
## Evaluation Metrics / 평가지표 / 评估指标

알고리즘 설계서의 4대 검증 지표를 정량적으로 측정합니다.
定量测量算法设计书的4项验证指标。

| # | 지표 / 指标 / Metric | 측정 방법 / 测量方法 |
|---|----------------------|---------------------|
| ① | 문서 검출 성공률 / 检测成功率 |  — 성공 수 / 전체 수 × 100% |
| ② | 처리 시간 / 处理时间 |  —  측정 (초) |
| ③ | 대비 향상도 / 对比增强度 | PSNR (dB), Laplacian Variance (전/후), SSIM (0~1) |
| ④ | 처리 전후 비교 / 前后对比 |  — 원본/교정/결과 시각화 |


In [ ]:
# 평가 지표 모듈 / 评估指标模块
# Evaluation Metrics Module
#
# 알고리즘 설계서 검증 지표 4종 구현:
# 实现算法设计书中4项验证指标:
#   1. 문서 영역 검출 성공률 / 文档检测成功率 — Detection Success Rate
#   2. 처리 시간 / 处理时间 — Processing Time
#   3. 처리 전후 대비 향상 정도 / 处理前后对比量化 — PSNR, Laplacian, SSIM
#   4. 처리 전후 이미지 비교 / 处理前后图像对比 — Visual Comparison
#
# Google Colab에는 scikit-image가 기본 설치되어 있습니다.
# Google Colab 已预装 scikit-image。
# 만약 누락된 경우 / 如有缺失:
# !pip install scikit-image pandas

from skimage.metrics import structural_similarity as ssim
import pandas as pd

print("Evaluation libraries loaded. / 평가 라이브러리 로드 완료. / 评估库已加载。")


# ===========================================================================
# 1. Laplacian Variance — 이미지 선명도 / 图像清晰度 / Image Sharpness
# ===========================================================================

def laplacian_variance(img):
    """Calculate image sharpness using Laplacian variance.
    Laplacian 분산값으로 이미지 선명도를 계산합니다.
    使用Laplacian方差计算图像清晰度。

    값이 높을수록 선명한 이미지 / 值越高图像越清晰 / Higher = sharper
    Uses: cv2.Laplacian()
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(lap.var())


# ===========================================================================
# 2. PSNR — 신호 대 잡음비 / 峰值信噪比 / Peak Signal-to-Noise Ratio
# ===========================================================================

def compute_psnr(img_a, img_b):
    """Calculate PSNR between two images. Resizes img_b if sizes differ.
    두 이미지 간 PSNR(Peak Signal-to-Noise Ratio)을 계산합니다.
    计算两幅图像之间的PSNR(峰值信噪比)。

    값이 높을수록 화질 손실이 적음 (일반적으로 30dB 이상이면 양호).
    值越高表示画质损失越小(通常30dB以上为良好)。
    Higher PSNR = less quality loss (generally >30dB is good).

    Args:
        img_a: Reference (BGR or grayscale) / 기준 이미지 / 参考图像
        img_b: Comparison (BGR or grayscale) / 비교 이미지 / 比较图像
    Returns:
        float: PSNR value in dB / PSNR 값 (dB) / PSNR值(dB)
    Uses: cv2.PSNR() or manual MSE calculation
    """
    if len(img_a.shape) == 3:
        a_gray = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
    else:
        a_gray = img_a.astype(np.float64)

    if len(img_b.shape) == 3:
        b_gray = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)
    else:
        b_gray = img_b.astype(np.float64)

    if a_gray.shape != b_gray.shape:
        b_gray = cv2.resize(b_gray, (a_gray.shape[1], a_gray.shape[0]))

    # MSE → PSNR / MSE 계산 후 PSNR 산출 / 计算MSE后求PSNR
    mse = np.mean((a_gray.astype(np.float64) - b_gray.astype(np.float64)) ** 2)
    if mse == 0:
        return float("inf")
    return float(20 * np.log10(255.0 / np.sqrt(mse)))


# ===========================================================================
# 3. SSIM — 구조적 유사도 / 结构相似度 / Structural Similarity
# ===========================================================================

def compute_ssim(img_a, img_b):
    """Compute SSIM between two images. Resizes img_b if sizes differ.
    두 이미지 간 SSIM을 계산합니다. 크기가 다르면 img_b를 img_a 크기로 조정합니다.
    计算两幅图像之间的SSIM。如果尺寸不同，将img_b调整为img_a的尺寸。

    SSIM ∈ [0, 1], 1에 가까울수록 원본과 구조적으로 유사.
    SSIM ∈ [0, 1], 越接近1表示与原图结构越相似。

    Uses: skimage.metrics.structural_similarity
    """
    if len(img_a.shape) == 3:
        a_gray = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
    else:
        a_gray = img_a

    if len(img_b.shape) == 3:
        b_gray = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)
    else:
        b_gray = img_b

    if a_gray.shape != b_gray.shape:
        b_gray = cv2.resize(b_gray, (a_gray.shape[1], a_gray.shape[0]))

    return float(ssim(a_gray, b_gray, data_range=b_gray.max() - b_gray.min()))


# ===========================================================================
# 4. 단일 이미지 종합 평가 / 单张图像综合评估
# ===========================================================================

def evaluate_result(pipeline_result, image_name="image"):
    """Evaluate a single pipeline result — all 4 verification metrics.
    단일 파이프라인 결과를 알고리즘 설계서의 4대 검증 지표로 평가합니다.
    使用算法设计书的4项验证指标评估单个流水线结果。

    평가 항목 / 评估项目:
      ① 문서 검출 성공 여부 / 文档检测是否成功
      ② 처리 시간 (초) / 处理时间(秒)
      ③ 대비 향상도: PSNR(dB), Laplacian 분산(전/후), SSIM
      ④ 처리 전후 시각 비교 (show_comparison 별도 호출)

    Args:
        pipeline_result: run_pipeline() 반환 dict
        image_name: 이미지 이름 (표시용)
    Returns:
        dict: {image_name, success, processing_time, psnr_db,
               laplacian_before, laplacian_after, laplacian_ratio, ssim_value}
    """
    success = pipeline_result["success"]
    proc_time = pipeline_result["processing_time"]
    original = pipeline_result["original"]
    result = pipeline_result["result"]

    # ③ 대비 향상도 계산 / 计算对比增强度
    lap_before = laplacian_variance(original)
    lap_after = laplacian_variance(result)
    lap_ratio = lap_after / lap_before if lap_before > 0 else 0.0
    psnr_val = compute_psnr(original, result)
    ssim_val = compute_ssim(original, result)

    status_kr = "성공" if success else "실패"
    status_cn = "成功" if success else "失败"

    # 출력 / 输出
    print("=" * 58)
    print(f"  EVALUATION REPORT / 평가 보고서 / 评估报告: {image_name}")
    print("=" * 58)
    print(f"  ① Detection / 문서 검출 / 文档检测 : {status_kr} ({status_cn})")
    print(f"  ② Time / 처리 시간 / 处理时间       : {proc_time:.3f}s")
    print(f"  ③ Quality / 화질 향상도 / 画质增强   :")
    print(f"       PSNR                        : {psnr_val:>8.2f} dB")
    print(f"       Laplacian (Before / 전 / 前) : {lap_before:>8.2f}")
    print(f"       Laplacian (After  / 후 / 后) : {lap_after:>8.2f}")
    print(f"       Laplacian Ratio (향상배율)    : {lap_ratio:>8.2f}x")
    print(f"       SSIM (구조유사도/结构相似度)    : {ssim_val:>8.4f}")
    print(f"  ④ Visual comparison → run: show_comparison(result, '{image_name}')")
    print("=" * 58)

    return {
        "image_name": image_name,
        "success": success,
        "processing_time": proc_time,
        "psnr_db": psnr_val,
        "laplacian_before": lap_before,
        "laplacian_after": lap_after,
        "laplacian_ratio": lap_ratio,
        "ssim_value": ssim_val,
    }


# ===========================================================================
# 5. 다중 이미지 평가 요약 + 종합 보고서 / 多图评估汇总+综合报告
# ===========================================================================

def evaluate_multiple(results_list):
    """Generate comprehensive evaluation summary for multiple images.
    여러 이미지의 평가 결과를 종합 요약 보고서로 출력합니다.
    生成多张图像的综合评估汇总报告。

    TR 고찰/분석용 4대 지표를 모두 포함:
    涵盖TR考察/分析所需的全部4项指标:
      ① 문서 검출 성공률 (Detection Success Rate)
      ② 평균 처리 시간 (Average Processing Time)
      ③ 평균 화질 향상도: PSNR, Laplacian Ratio, SSIM
      ④ 처리 결과 요약 테이블 (Summary Table)

    Args:
        results_list: list of dicts from evaluate_result()
    Returns:
        pd.DataFrame
    """
    df = pd.DataFrame(results_list)
    n = len(df)

    # ── 개별 결과 테이블 / 个别结果表 ──
    df_display = df.rename(columns={
        "image_name": "Image / 이미지 / 图像",
        "success": "Detected / 검출 / 检测",
        "processing_time": "Time(s) / 시간 / 时间",
        "psnr_db": "PSNR(dB) / 화질 / 画质",
        "laplacian_before": "Lap Before / 전 / 前",
        "laplacian_after": "Lap After / 후 / 后",
        "laplacian_ratio": "Lap Ratio / 향상배율 / 倍率",
        "ssim_value": "SSIM / 유사도 / 相似度",
    })

    # ── 종합 보고서 출력 / 综合报告输出 ──
    print()
    print("=" * 80)
    print("  COMPREHENSIVE EVALUATION REPORT / 종합 평가 보고서 / 综合评估报告")
    print("=" * 80)
    print(f"  Total images / 총 이미지 수 / 图像总数: {n}")
    print()
    print(df_display.to_string(index=False))
    print()

    # ── ① 문서 검출 성공률 ──
    success_count = df["success"].sum()
    success_rate = success_count / n * 100
    print(f"  ┌─ ① Detection Success Rate / 문서 검출 성공률 / 检测成功率")
    print(f"  │   {success_count}/{n} = {success_rate:.1f}%")

    # ── ② 처리 시간 통계 ──
    avg_time = df["processing_time"].mean()
    min_time = df["processing_time"].min()
    max_time = df["processing_time"].max()
    print(f"  ├─ ② Processing Time / 처리 시간 / 处理时间")
    print(f"  │   Mean / 평균 / 平均: {avg_time:.3f}s  |  Min: {min_time:.3f}s  |  Max: {max_time:.3f}s  |  Total / 합계 / 总计: {df["processing_time"].sum():.3f}s")

    # ── ③ 대비 향상도 통계 ──
    avg_psnr = df["psnr_db"].replace(float("inf"), np.nan).mean()
    avg_lap_before = df["laplacian_before"].mean()
    avg_lap_after = df["laplacian_after"].mean()
    avg_lap_ratio = df["laplacian_ratio"].mean()
    avg_ssim = df["ssim_value"].mean()
    print(f"  ├─ ③ Quality Enhancement / 화질 향상도 / 画质增强度")
    print(f"  │   PSNR          (Mean / 평균 / 平均): {avg_psnr:>8.2f} dB  (↑ higher=less loss / 높을수록 손실 적음 / 越高损失越小)")
    print(f"  │   Laplacian 전/前  (Mean / 평균 / 平均): {avg_lap_before:>8.2f}")
    print(f"  │   Laplacian 후/后  (Mean / 평균 / 平均): {avg_lap_after:>8.2f}")
    print(f"  │   Laplacian Ratio  (Mean / 평균 / 平均): {avg_lap_ratio:>8.2f}x  (↑ >1.0 = 선명도 향상 / 清晰度提升)")
    print(f"  │   SSIM            (Mean / 평균 / 平均): {avg_ssim:>8.4f}   (↑ 1.0 = identical / 동일 / 相同)")

    # ── ④ 시각 비교 안내 ──
    print(f"  └─ ④ Visual Comparison / 시각 비교 / 视觉对比")
    print(f"      → Run: show_comparison(pipeline_result, image_name) for each image")
    print(f"      → 또는 Cell 10 (Visualize Results) 실행 / 或运行单元格10")
    print("=" * 80)

    return df


# ===========================================================================
# 6. 처리 전후 시각 비교 / 处理前后视觉对比
# ===========================================================================

def show_comparison(pipeline_result, image_name="image"):
    """Side-by-side visual comparison with quality scores.
    원본/교정/결과를 화질 점수와 함께 나란히 시각 비교합니다.
    并排显示原图/矫正图/结果图并标注画质评分。

    Displays: Original → Perspective Corrected (if success) → Enhanced Final
    각 이미지에 Laplacian 점수, 최종 결과에는 SSIM과 PSNR도 함께 표시.

    Uses: show_images()
    """
    original = pipeline_result["original"]
    result = pipeline_result["result"]
    success = pipeline_result["success"]
    warped = pipeline_result.get("warped")

    lap_orig = laplacian_variance(original)
    lap_result = laplacian_variance(result)
    ssim_val = compute_ssim(original, result)
    psnr_val = compute_psnr(original, result)

    images = []
    titles = []

    # 원본 / 原图
    images.append(original)
    titles.append(f"① Original / 원본 / 原图\n[{image_name}]\nLaplacian: {lap_orig:.1f}")

    # 투시 보정 (검출 성공 시만) / 透视矫正(仅检测成功时)
    if warped is not None and success:
        lap_warped = laplacian_variance(warped)
        images.append(warped)
        titles.append(f"② Perspective Corrected / 투시 보정 / 透视矫正\nLaplacian: {lap_warped:.1f}")

    # 최종 향상 결과 / 最终增强结果
    images.append(result)
    titles.append(f"③ Enhanced Final / 최종 향상 / 增强结果\nLaplacian: {lap_result:.1f}\nPSNR: {psnr_val:.2f}dB | SSIM: {ssim_val:.4f}")

    show_images(images, titles, cols=len(images), figsize=(6 * len(images), 5))


print("Evaluation functions defined. / 평가 함수 정의 완료. / 评估函数已定义。")
print("-" * 55)
print("Usage example / 사용 예시 / 使用示例:")
print("  res = run_pipeline('doc.jpg')")
print("  metrics = evaluate_result(res, image_name='test1')                     # 단일 평가 / 单张评估")
print("  show_comparison(res, image_name='test1')                              # 시각 비교 / 视觉对比")
print()
print("  # Multiple images / 여러 이미지 / 多张图像:")
print("  all_metrics = [evaluate_result(r, n) for r, n in zip(results, names)]")
print("  df = evaluate_multiple(all_metrics)                                   # 종합 보고서 / 综合报告")


---
## 8. Upload Document Images / 문서 이미지 업로드 / 上传文档图像

아래 셀을 실행하여 이미지를 업로드하고 처리하세요.
运行下方单元格上传图像并进行处理。

### 옵션 A / 选项A: 직접 업로드 (권장 / 推荐)
아래 버튼을 클릭하여 이미지 파일을 선택하세요. 여러 장 선택 가능합니다.
点击下方按钮选择图像文件，可多选。

In [ ]:
# 8-A. 이미지 업로드 / 图像上传 (지원 형식 / 支持格式: JPG, PNG, BMP)

print("이미지 파일을 선택하세요 (여러 장 가능)... / 请选择图像文件（可多选）...")
uploaded = files.upload()

input_images = {}
for fname, data in uploaded.items():
    if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
        # Save uploaded file / 업로드된 파일 저장 / 保存上传文件
        with open(fname, 'wb') as f:
            f.write(data)
        img = cv2.imread(fname)
        if img is not None:
            input_images[fname] = img
            print(f"  [OK] {fname} — {img.shape[1]}x{img.shape[0]}")
        else:
            print(f"  [FAIL/실패/失败] {fname} — could not read as image / 이미지 읽기 실패 / 无法读取为图像")
    else:
        print(f"  [SKIP/건너뜀/跳过] {fname} — unsupported format / 지원하지 않는 형식 / 不支持的格式")

if not input_images:
    print("\n⚠️ No valid images uploaded. Please upload JPG, PNG, or BMP files.")
    print("   유효한 이미지가 없습니다. JPG, PNG, BMP 파일을 업로드하세요.")
    print("   没有有效图像，请上传 JPG、PNG 或 BMP 文件。")
else:
    print(f"\n✅ {len(input_images)} image(s) ready for processing. / 처리 준비 완료. / 图像已准备好处理。")

### 옵션 B / 选项B: Google Drive에서 이미지 가져오기 / 从Google Drive获取图像
Google Drive를 마운트하고 이미지 경로를 지정하세요.
挂载Google Drive并指定图像路径。

In [ ]:
# 8-B. (선택 / 可选) Google Drive 마운트 / 挂载Google Drive
# 필요한 경우에만 아래 코드를 실행하세요.
# 仅在需要时运行以下代码。

use_drive = False  # True로 변경하여 Google Drive 사용 / 改为True以使用Google Drive

if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # 이미지가 있는 폴더 경로 (자신의 경로로 변경하세요)
    # 图像所在文件夹路径（请改为自己的路径）
    DRIVE_IMAGE_DIR = "/content/drive/MyDrive/document_images"
    
    if os.path.isdir(DRIVE_IMAGE_DIR):
        input_images = {}
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            for fpath in glob.glob(os.path.join(DRIVE_IMAGE_DIR, ext)):
                fpath_lower = fpath.lower()
                if fpath_lower.endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img = cv2.imread(fpath)
                    if img is not None:
                        fname = os.path.basename(fpath)
                        input_images[fname] = img
                        print(f"  [OK] {fname}")
        print(f"\n✅ {len(input_images)} image(s) loaded from Drive. / Drive에서 로드 완료. / 从Drive加载完成。")
    else:
        print(f"⚠️ Directory not found / 디렉터리 없음 / 目录不存在: {DRIVE_IMAGE_DIR}")
        print("Please update DRIVE_IMAGE_DIR to your actual image folder path.")
        print("DRIVE_IMAGE_DIR를 실제 이미지 폴더 경로로 변경하세요.")
        print("请将DRIVE_IMAGE_DIR更新为您实际的图像文件夹路径。")

---
## 9. Run Pipeline / 파이프라인 실행 / 运行流水线

업로드한 모든 이미지를 처리하고 결과를 출력합니다.
处理所有上传的图像并输出结果。

In [ ]:
# 9. 업로드된 모든 이미지에 대해 파이프라인 실행
# 9. 对所有上传的图像运行流水线

if 'input_images' not in dir() or not input_images:
    print("⚠️ 먼저 이미지를 업로드하세요! (옵션 A 또는 B 실행)")
    print("   请先上传图像！（运行选项A或B）")
else:
    OUTPUT_DIR = "output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    success_count = 0
    total_time = 0.0
    all_results = []
    
    for fname, img in input_images.items():
        basename = os.path.splitext(fname)[0]
        print(f"{'='*60}")
        print(f"[INFO] Processing / 처리 중 / 处理中: {fname} ({img.shape[1]}x{img.shape[0]})")
        
        result = run_pipeline(img, OUTPUT_DIR, save_steps=True, basename=basename)
        
        print(f"[INFO] Document detected / 문서 검출 / 文档检测: {result['success']}")
        print(f"[INFO] Processing time / 처리 시간 / 处理时间: {result['processing_time']:.3f}s")
        print(f"[INFO] Saved / 저장됨 / 已保存: {result['output_path']}")
        
        if result['success']:
            success_count += 1
        total_time += result['processing_time']
        all_results.append((basename, result))
    
    total = len(input_images)
    print(f"\n{'='*60}")
    print(f"[SUMMARY/요약/总结] Detection success / 검출 성공 / 检测成功: {success_count}/{total} ({100*success_count/total:.1f}%)")
    print(f"[SUMMARY/요약/总结] Total processing time / 총 처리 시간 / 总处理时间: {total_time:.3f}s")
    print(f"[SUMMARY/요약/总结] Average time / 평균 시간 / 平均时间: {total_time/total:.3f}s/image")

---
## 10. Visualize Results / 결과 시각화 / 结果可视化

각 이미지의 단계별 중간 결과를 확인합니다.
查看每张图像的各步骤中间结果。

In [ ]:
# 10. 결과 시각화 / 结果可视化

if 'all_results' not in dir() or not all_results:
    print("⚠️ 먼저 파이프라인을 실행하세요! / 请先运行流水线！")
else:
    for basename, result in all_results:
        print(f"\n{'='*60}")
        status_emoji = "✅" if result['success'] else "⚠️"
        print(f"{status_emoji} {basename} — {result['processing_time']:.3f}s")
        
        # Original vs Result comparison / 원본 vs 결과 비교 / 原图vs结果对比
        original = result['original']
        final = result['result']
        warped = result.get('warped')
        
        images = [original]
        titles = ["01. Original / 원본 / 原图"]
        
        if warped is not None and result['success']:
            images.append(warped)
            titles.append("06. Perspective Corrected / 투시 보정 / 透视矫正")
        
        images.append(final)
        titles.append("07. Enhanced (Final) / 향상 결과 / 增强结果")
        
        show_images(images, titles, cols=len(images), figsize=(6*len(images), 5))
        
        # Show all 7 steps if available / 7단계 전체 표시 / 显示全部7步
        steps = result.get('steps_images', {})
        if steps:
            step_names = ['01_original', '02_grayscale', '03_blurred', 
                         '04_edges', '05_contours', '06_perspective', '07_enhanced']
            step_titles_kr = ['원본', '그레이스케일', '가우시안 블러', 
                            '에지 검출', '윤곽선 검출', '투시 변환', '향상 결과']
            step_titles_cn = ['原图', '灰度化', '高斯模糊', 
                            '边缘检测', '轮廓检测', '透视变换', '增强结果']
            step_images = [steps[name] for name in step_names if name in steps]
            step_titles = [f"{name.replace('_',' ').title()}\n{kr}\n{cn}" 
                          for name, kr, cn in zip(step_names, step_titles_kr, step_titles_cn) 
                          if name in steps]
            if step_images:
                print(f"\n  --- All 7 Steps for {basename} / 전체 7단계 / 全部7步骤 ---")
                show_images(step_images, step_titles, cols=4, figsize=(16, 10))

---
## 11. Download Results / 결과 다운로드 / 下载结果

처리된 이미지와 중간 결과를 ZIP 파일로 다운로드합니다.
将处理后的图像和中间结果下载为ZIP文件。

In [ ]:
# 11. 결과 다운로드 / 下载结果

import shutil

if os.path.isdir(OUTPUT_DIR if 'OUTPUT_DIR' in dir() else 'output'):
    output_zip = 'document_correction_results.zip'
    shutil.make_archive('document_correction_results', 'zip', 
                        OUTPUT_DIR if 'OUTPUT_DIR' in dir() else 'output')
    files.download(output_zip)
    print(f"✅ Results downloaded / 결과 다운로드 완료 / 结果已下载: {output_zip}")
else:
    print("⚠️ No output directory found. Run the pipeline first.")
    print("   출력 디렉터리가 없습니다. 파이프라인을 먼저 실행하세요.")
    print("   未找到输出目录，请先运行流水线。")

---
## 12. 사용 방법 요약 / 使用说明 / Usage Summary

### 실행 순서 / 执行顺序

| 순서 | 내용 | 内容 |
|------|------|------|
| 1 | 환경 설정 / 环境设置 | 상단 코드 셀 순서대로 실행 / 按顺序运行上方代码单元格 (`Runtime → Run all`) |
| 2 | 이미지 업로드 / 上传图像 | 셀 8-A 실행 후 이미지 파일 선택 (JPG/PNG/BMP) / 运行单元格8-A后选择图像文件 |
| 3 | 처리 실행 / 执行处理 | 셀 9에서 모든 이미지 자동 처리 / 单元格9自动处理所有图像 |
| 4 | 결과 확인 / 查看结果 | 셀 10에서 단계별 결과 시각화 / 单元格10中查看各步骤结果可视化 |
| 5 | 다운로드 / 下载 | 셀 11에서 ZIP 파일 다운로드 / 单元格11下载ZIP文件 |

### 처리 과정 / 处理步骤

| Step | 한국어 | 中文 | English |
|------|--------|------|---------|
| 01 | 원본 이미지 | 原始图像 | Original Image |
| 02 | 그레이스케일 변환 | 灰度转换 | Grayscale Conversion |
| 03 | 가우시안 블러 | 高斯模糊 | Gaussian Blur |
| 04 | 케니 에지 검출 | Canny边缘检测 | Canny Edge Detection |
| 05 | 윤곽선 검출 | 轮廓检测 | Contour Detection |
| 06 | 투시 변환 | 透视变换 | Perspective Transform |
| 07 | 향상 결과 | 增强结果 | Enhanced Result |

### 파라미터 조정 / 参数调整

검출 성능을 개선하려면 아래 파라미터를 조정해보세요.
要改善检测性能，请尝试调整以下参数。

- Canny thresholds: `detect_edges(blurred, low=50, high=150)`
- Gaussian kernel: `apply_gaussian_blur(gray, kernel_size=(7,7))`
- CLAHE clip limit: `apply_clahe(gray, clip_limit=3.0)`

---
## C1. Batch Evaluation Summary / 배치 평가 요약 / 批量评估汇总

모든 이미지의 PSNR / SSIM / Laplacian Ratio를 자동 집계하고 pandas로 예쁜 표를 출력합니다.
自动汇总所有图像的 PSNR / SSIM / Laplacian Ratio，用pandas生成漂亮表格。
TR에 바로 스크린샷으로 사용 가능 / 可直接截图放入TR。


In [ ]:
# C1. 배치 평가 요약 / 批量评估汇总
# 모든 업로드 이미지를 자동 처리 + 평가 + 요약 / 自动处理所有上传图像并评估汇总

def run_batch_evaluation(input_images, output_dir="output"):
    """Run pipeline + evaluation on all images, generate summary table.
    모든 이미지에 대해 pipeline + 평가를 실행하고 요약 테이블을 생성합니다.
    对所有图像运行pipeline+评估并生成汇总表。

    Args:
        input_images: dict of {filename: BGR image} / {파일명: BGR이미지} 사전 / 字典
        output_dir: output directory / 출력 디렉터리 / 输出目录
    Returns:
        pd.DataFrame with styled formatting
    """
    os.makedirs(output_dir, exist_ok=True)
    all_metrics = []

    for fname, img in input_images.items():
        basename = os.path.splitext(fname)[0]
        print(f"Processing: {fname} ...", end=" ")
        pipe_res = run_pipeline(img, output_dir, save_steps=True, basename=basename)
        m = evaluate_result(pipe_res, image_name=basename)
        all_metrics.append(m)

    # ── 종합 보고서 / 综合报告 ──
    df = evaluate_multiple(all_metrics)

    # ── CSV 저장 / 保存CSV ──
    csv_path = os.path.join(output_dir, "evaluation_summary.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\n[CSV] Saved / 저장됨 / 已保存: {csv_path}")

    # ── Styled HTML table (TR 스크린샷용 / 用于TR截图) ──
    def highlight_psnr(val):
        """PSNR > 15 = green (good), < 8 = red (poor)"""
        try:
            v = float(val)
            if v > 15:
                return "background-color: #c8e6c9"
            elif v < 8:
                return "background-color: #ffcdd2"
        except (ValueError, TypeError):
            pass
        return ""

    def highlight_ssim(val):
        """SSIM > 0.5 = green, < 0.2 = red"""
        try:
            v = float(val)
            if v > 0.5:
                return "background-color: #c8e6c9"
            elif v < 0.2:
                return "background-color: #ffcdd2"
        except (ValueError, TypeError):
            pass
        return ""

    styled = (df.style
               .applymap(highlight_psnr, subset=["psnr_db"])
               .applymap(highlight_ssim, subset=["ssim_value"])
               .format({
                   "psnr_db": "{:.2f} dB",
                   "laplacian_before": "{:.1f}",
                   "laplacian_after": "{:.1f}",
                   "laplacian_ratio": "{:.2f}x",
                   "ssim_value": "{:.4f}",
                   "processing_time": "{:.3f}s",
               })
               .set_caption("Evaluation Summary / 평가 요약 / 评估汇总"))

    print("\n[TR TABLE / TR용 표 / TR表格]:")
    display(styled)

    return df


# ── 실행 / 执行 ──
print("C1: Batch Evaluation Summary")
print("=" * 55)
if 'input_images' in dir() and input_images:
    df_summary = run_batch_evaluation(input_images, OUTPUT_DIR if 'OUTPUT_DIR' in dir() else "output")
else:
    print("No input_images found. Please upload images first (Cell 8-A).")
    print("input_images가 없습니다. 먼저 이미지를 업로드하세요 (Cell 8-A).")
    print("未找到input_images。请先上传图像（Cell 8-A）。")


---
## C2. Enhanced Comparison Figure / 비교도 시각화 / 增强对比图

고해상도 3단 비교도를 생성합니다. 보고서/PPT에 바로 사용 가능.
生成高分辨率三联对比图，可直接用于报告/PPT。


In [ ]:
# C2. 고해상도 비교도 생성 / 高清对比图生成

def generate_comparison_figure(pipeline_result, image_name="image", save_path=None, dpi=200):
    """Generate polished 3-panel comparison figure for reports/PPT.
    보고서/PPT용 고품질 3단 비교도를 생성합니다.
    生成用于报告/PPT的高质量三联对比图。

    Panels / 패널 / 面板:
      Left / 좌 / 左 : Original + Laplacian score
      Center / 중 / 中: Perspective Corrected + Laplacian score (success only)
      Right / 우 / 右 : Enhanced Final + PSNR + SSIM + Laplacian

    Args:
        pipeline_result: run_pipeline() 반환 dict
        image_name: image label / 이미지 이름 / 图像名称
        save_path: if set, saves to file / 지정 시 파일 저장 / 指定则保存文件
        dpi: output resolution / 출력 해상도 / 输出分辨率
    """
    original = pipeline_result["original"]
    result_img = pipeline_result["result"]
    success = pipeline_result["success"]
    warped = pipeline_result.get("warped")

    # ── 지표 계산 / 计算指标 ──
    lap_orig = laplacian_variance(original)
    lap_result = laplacian_variance(result_img)
    lap_ratio = lap_result / lap_orig if lap_orig > 0 else 0
    psnr_val = compute_psnr(original, result_img)
    ssim_val = compute_ssim(original, result_img)

    # ── 3-panel layout / 3패널 레이아웃 / 三联布局 ──
    n_cols = 3 if (warped is not None and success) else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 7), dpi=dpi)
    if n_cols == 1:
        axes = [axes]

    # Panel 1: Original
    ax = axes[0]
    ax.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    ax.set_title(f"① Original / 원본 / 原图\nLaplacian: {lap_orig:.1f}", fontsize=13, fontweight="bold")
    ax.axis("off")
    ax.text(10, 30, f"Lap: {lap_orig:.1f}", fontsize=11, color="white",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.6))

    # Panel 2: Perspective Corrected (if success)
    panel_idx = 1
    if warped is not None and success:
        lap_warped = laplacian_variance(warped)
        ax = axes[1]
        ax.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
        ax.set_title(f"② Perspective Corrected / 투시 보정 / 透视矫正\nLaplacian: {lap_warped:.1f}", fontsize=13, fontweight="bold")
        ax.axis("off")
        ax.text(10, 30, f"Lap: {lap_warped:.1f}", fontsize=11, color="white",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.6))
        panel_idx = 2

    # Panel 3: Enhanced Final
    ax = axes[panel_idx]
    ax.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"③ Enhanced Final / 최종 향상 / 增强结果\nLaplacian: {lap_result:.1f}  |  PSNR: {psnr_val:.2f}dB  |  SSIM: {ssim_val:.4f}",
                fontsize=13, fontweight="bold")
    ax.axis("off")
    ax.text(10, 30, f"Lap: {lap_result:.1f}  ({lap_ratio:.1f}x)\nPSNR: {psnr_val:.1f}dB\nSSIM: {ssim_val:.4f}",
            fontsize=10, color="white",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7))

    fig.suptitle(f"Document Correction Result / 문서 교정 결과 / 文档矫正结果 — {image_name}",
                fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()

    # ── 저장 / 保存 ──
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")
        print(f"Figure saved / 그림 저장됨 / 图片已保存: {save_path}")

    plt.show()


# ── 실행 예시 / 执行示例 ──
print("C2: Enhanced Comparison Figure")
print("=" * 55)
if 'all_results' in dir() and all_results:
    # 첫 번째 이미지에 대해 비교도 생성 / 对第一张图生成对比图
    basename, res = all_results[0]
    out_dir = OUTPUT_DIR if 'OUTPUT_DIR' in dir() else "output"
    save_p = os.path.join(out_dir, f"{basename}_comparison.png")
    generate_comparison_figure(res, image_name=basename, save_path=save_p)
    print(f"\nTo generate for all images / 모든 이미지에 대해 생성 / 为所有图像生成:")
    print("  for name, r in all_results:")
    print("      generate_comparison_figure(r, image_name=name, save_path=f'output/{name}_comparison.png')")
else:
    print("No results found. Please run pipeline first (Cell 9).")
    print("all_results가 없습니다. 먼저 파이프라인을 실행하세요 (Cell 9).")
    print("未找到all_results。请先运行流水线（Cell 9）。")


---
## C3. Parameter Sensitivity Experiment / 파라미터 민감도 실험 / 参数敏感性实验

Canny threshold, Gaussian kernel, CLAHE clip limit 변화에 따른 영향을 실험합니다.
测试 Canny阈值、Gaussian核大小、CLAHE clip limit 对结果的影响。
TR의 "실험 분석" 섹션에 활용 / 用于TR"实验分析"部分。


In [ ]:
# C3. 파라미터 민감도 실험 / 参数敏感性实验

def _run_pipeline_with_params(img, canny_low, canny_high, gauss_ksize, clahe_clip):
    """Run pipeline with custom parameters, return metrics only (no save).
    사용자 지정 파라미터로 pipeline을 실행하고 지표만 반환합니다.
    使用自定义参数运行pipeline并仅返回指标。"""
    import time as _time
    t0 = _time.time()
    original = img.copy()
    gray = to_grayscale(original)
    blurred = apply_gaussian_blur(gray, kernel_size=gauss_ksize)
    edges = detect_edges(blurred, low=canny_low, high=canny_high)
    dilated = dilate_edges(edges)
    contours = find_contours(dilated)
    img_area = original.shape[0] * original.shape[1]
    doc_contour = find_document_contour(contours, img_area)

    if doc_contour is not None:
        ordered_pts = order_points(doc_contour)
        warped = apply_perspective_transform(original, ordered_pts)
    else:
        warped = original

    # Enhancement with custom CLAHE
    gray_w = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    clahed = apply_clahe(gray_w, clip_limit=clahe_clip)
    thresh = apply_adaptive_threshold(clahed)
    cleaned = apply_morphology(thresh)
    result = cv2.cvtColor(cleaned, cv2.COLOR_GRAY2BGR)

    return {
        "canny_low": canny_low, "canny_high": canny_high,
        "gauss_ksize": f"{gauss_ksize[0]}x{gauss_ksize[1]}",
        "clahe_clip": clahe_clip,
        "detected": doc_contour is not None,
        "time": _time.time() - t0,
        "lap_before": laplacian_variance(original),
        "lap_after": laplacian_variance(result),
        "lap_ratio": laplacian_variance(result) / max(laplacian_variance(original), 1e-6),
        "psnr": compute_psnr(original, result),
        "ssim": compute_ssim(original, result),
    }


def run_sensitivity_experiment(image, output_dir="output"):
    """Run parameter sensitivity experiment across 4 parameter groups.
    4개 파라미터 그룹에 대해 민감도 실험을 실행합니다.
    对4组参数执行敏感性实验。

    Returns:
        dict with keys: canny_low, canny_high, gaussian, clahe
        each value is a pd.DataFrame of results
    """
    os.makedirs(output_dir, exist_ok=True)
    all_results = {}

    # ── Experiment 1: Canny low threshold / 케니 하한 임계값 / Canny低阈值 ──
    print("\n" + "=" * 60)
    print("  Experiment 1: Canny Low Threshold / 케니 low / Canny低阈值")
    print("=" * 60)
    rows = []
    for low in [50, 75, 100, 125]:
        r = _run_pipeline_with_params(image, canny_low=low, canny_high=200,
                                       gauss_ksize=(5,5), clahe_clip=2.0)
        rows.append(r)
        print(f"  low={low:>3d}: detected={r['detected']}, PSNR={r['psnr']:.1f}dB, SSIM={r['ssim']:.4f}, LapRatio={r['lap_ratio']:.1f}x")
    all_results["canny_low"] = pd.DataFrame(rows)

    # ── Experiment 2: Canny high threshold / 케니 상한 임계값 / Canny高阈值 ──
    print("\n" + "=" * 60)
    print("  Experiment 2: Canny High Threshold / 케니 high / Canny高阈值")
    print("=" * 60)
    rows = []
    for high in [150, 200, 250, 300]:
        r = _run_pipeline_with_params(image, canny_low=75, canny_high=high,
                                       gauss_ksize=(5,5), clahe_clip=2.0)
        rows.append(r)
        print(f"  high={high:>3d}: detected={r['detected']}, PSNR={r['psnr']:.1f}dB, SSIM={r['ssim']:.4f}, LapRatio={r['lap_ratio']:.1f}x")
    all_results["canny_high"] = pd.DataFrame(rows)

    # ── Experiment 3: Gaussian kernel size / 가우시안 커널 크기 / 高斯核大小 ──
    print("\n" + "=" * 60)
    print("  Experiment 3: Gaussian Kernel / 가우시안 커널 / 高斯核")
    print("=" * 60)
    rows = []
    for ks in [(3,3), (5,5), (7,7), (9,9)]:
        r = _run_pipeline_with_params(image, canny_low=75, canny_high=200,
                                       gauss_ksize=ks, clahe_clip=2.0)
        rows.append(r)
        print(f"  kernel={ks}: detected={r['detected']}, PSNR={r['psnr']:.1f}dB, SSIM={r['ssim']:.4f}, LapRatio={r['lap_ratio']:.1f}x")
    all_results["gaussian"] = pd.DataFrame(rows)

    # ── Experiment 4: CLAHE clip limit / CLAHE 클립 한계 / CLAHE裁剪限值 ──
    print("\n" + "=" * 60)
    print("  Experiment 4: CLAHE clipLimit / CLAHE clipLimit / CLAHE裁剪限值")
    print("=" * 60)
    rows = []
    for clip in [1.0, 2.0, 3.0, 4.0, 5.0]:
        r = _run_pipeline_with_params(image, canny_low=75, canny_high=200,
                                       gauss_ksize=(5,5), clahe_clip=clip)
        rows.append(r)
        print(f"  clip={clip:.1f}: detected={r['detected']}, PSNR={r['psnr']:.1f}dB, SSIM={r['ssim']:.4f}, LapRatio={r['lap_ratio']:.1f}x")
    all_results["clahe"] = pd.DataFrame(rows)

    # ── Visualization: 2x2 comparison grid / 2x2 비교 그리드 / 2x2对比网格 ──
    print("\n" + "=" * 60)
    print("  Generating comparison charts / 비교 차트 생성 중 / 生成对比图...")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=150)
    exp_names = ["canny_low", "canny_high", "gaussian", "clahe"]
    exp_titles_kr = ["Canny Low Threshold", "Canny High Threshold", "Gaussian Kernel Size", "CLAHE clipLimit"]
    exp_titles_cn = ["Canny低阈值", "Canny高阈值", "高斯核大小", "CLAHE裁剪限值"]
    x_labels = ["low", "high", "kernel", "clip"]

    for idx, (exp_name, title_kr, title_cn, x_label) in enumerate(zip(exp_names, exp_titles_kr, exp_titles_cn, x_labels)):
        ax = axes[idx // 2][idx % 2]
        df = all_results[exp_name]
        x_vals = df.iloc[:, 0].astype(str).tolist()
        x_idx = range(len(x_vals))

        # Plot PSNR, SSIM*100, LapRatio on dual axis
        ax2 = ax.twinx()
        line1, = ax.plot(x_idx, df["psnr"].values, "b-o", linewidth=2, markersize=8, label="PSNR (dB)")
        line2, = ax.plot(x_idx, df["lap_ratio"].values, "g-s", linewidth=2, markersize=8, label="Laplacian Ratio")
        line3, = ax2.plot(x_idx, df["ssim"].values * 100, "r-^", linewidth=2, markersize=8, label="SSIM x100")

        # Mark detection status
        for i, detected in enumerate(df["detected"]):
            if not detected:
                ax.axvline(x=i, color="red", linestyle="--", alpha=0.3, linewidth=3)

        ax.set_xticks(x_idx)
        ax.set_xticklabels(x_vals)
        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel("PSNR (dB) / Laplacian Ratio", fontsize=10, color="blue")
        ax2.set_ylabel("SSIM x100", fontsize=10, color="red")
        ax.set_title(f"{title_kr} / {title_cn}", fontsize=13, fontweight="bold")
        ax.grid(True, alpha=0.3)

        # Combined legend
        lines = [line1, line2, line3]
        labels = [l.get_label() for l in lines]
        ax.legend(lines, labels, loc="upper left", fontsize=9)

    fig.suptitle("Parameter Sensitivity Analysis / 파라미터 민감도 분석 / 参数敏感性分析",
                fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()

    chart_path = os.path.join(output_dir, "sensitivity_analysis.png")
    fig.savefig(chart_path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Chart saved / 차트 저장됨 / 图表已保存: {chart_path}")
    plt.show()

    # ── CSV 저장 ──
    for exp_name, df in all_results.items():
        csv_p = os.path.join(output_dir, f"sensitivity_{exp_name}.csv")
        df.to_csv(csv_p, index=False, encoding="utf-8-sig")
    print(f"CSV files saved to / CSV 파일 저장됨 / CSV已保存: {output_dir}/")

    return all_results


# ── 실행 / 执行 ──
print("C3: Parameter Sensitivity Experiment")
print("=" * 55)
if 'input_images' in dir() and input_images:
    # 첫 번째 이미지로 실험 / 用第一张图做实验
    first_img = list(input_images.values())[0]
    first_name = list(input_images.keys())[0]
    print(f"Running sensitivity experiment on / 실험 대상 / 实验对象: {first_name}")
    sens_results = run_sensitivity_experiment(first_img, OUTPUT_DIR if 'OUTPUT_DIR' in dir() else "output")
    print("\n[CONCLUSION / 결론 / 结论]")
    print("  Review the charts above to determine optimal parameters.")
    print("  위 차트를 검토하여 최적 파라미터를 결정하세요.")
    print("  查看上方图表以确定最佳参数。")
else:
    print("No images found. Please upload first (Cell 8-A).")
    print("이미지가 없습니다. 먼저 업로드하세요 (Cell 8-A).")
    print("未找到图像。请先上传（Cell 8-A）。")


---
## C4. Full Pipeline Flow / 전체 파이프라인 흐름도 / 全流程中间步骤

대표 이미지 1장의 전체 처리 흐름을 한눈에 보여주는 대형 그림입니다.
将一张代表性图像的完整处理流程拼接为一张大图。
TR "실험 결과" 섹션에 바로 사용 가능 / 可直接用于TR"实验结果"部分。


In [ ]:
# C4. 전체 파이프라인 흐름도 / 全流程流程图

def generate_pipeline_flowchart(pipeline_result, image_name="image", save_path=None, dpi=180):
    """Generate a full pipeline flowchart showing all intermediate steps.
    모든 중간 단계를 포함한 전체 파이프라인 흐름도를 생성합니다.
    生成包含所有中间步骤的完整流水线流程图。

    8 steps / 8단계 / 8步骤:
      Original → Grayscale → Gaussian Blur → Canny Edges
      → Dilated Edges → Contours → Perspective → Enhanced

    Args:
        pipeline_result: run_pipeline() 반환 dict (must have save_steps=True)
        image_name: image label
        save_path: save path for the figure
        dpi: resolution
    """
    steps = pipeline_result.get("steps_images", {})
    if not steps:
        print("No step images found. Re-run pipeline with save_steps=True.")
        print("단계 이미지가 없습니다. save_steps=True로 파이프라인을 다시 실행하세요.")
        print("未找到步骤图像。请用save_steps=True重新运行流水线。")
        return

    # ── Assemble 8-panel figure / 8패널 그림 구성 / 组装8面板图 ──
    step_keys = ["01_original", "02_grayscale", "03_blurred", "04_edges",
                 "05_contours", "06_perspective", "07_enhanced"]
    step_titles_kr = ["① 원본\nOriginal", "② 그레이스케일\nGrayscale",
                      "③ 가우시안 블러\nGaussian Blur", "④ 케니 에지\nCanny Edges",
                      "⑤ 윤곽선 검출\nContours", "⑥ 투시 변환\nPerspective",
                      "⑦ 최종 향상\nEnhanced"]
    step_titles_cn = ["① 原图", "② 灰度化", "③ 高斯模糊", "④ Canny边缘",
                      "⑤ 轮廓检测", "⑥ 透视变换", "⑦ 最终增强"]

    valid_steps = [(k, kr, cn) for k, kr, cn in zip(step_keys, step_titles_kr, step_titles_cn) if k in steps]
    n_steps = len(valid_steps)
    cols = 4
    rows = (n_steps + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows), dpi=dpi)
    axes = axes.flatten() if rows > 1 else [axes] if rows == 1 else axes

    for i, (key, kr_title, cn_title) in enumerate(valid_steps):
        ax = axes[i]
        img = steps[key]
        if len(img.shape) == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{kr_title}\n{cn_title}", fontsize=11, fontweight="bold")
        ax.axis("off")

        # Add arrow between panels
        if i < len(valid_steps) - 1:
            ax.text(1.02, 0.5, "→", transform=ax.transAxes, fontsize=20,
                    ha="left", va="center", color="gray", fontweight="bold")

    # Hide unused subplots
    for j in range(n_steps, len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"Document Correction Pipeline / 문서 교정 파이프라인 / 文档矫正流水线 — {image_name}",
                fontsize=18, fontweight="bold", y=1.01)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")
        print(f"Flowchart saved / 흐름도 저장됨 / 流程图已保存: {save_path}")

    plt.show()


# ── 실행 / 执行 ──
print("C4: Full Pipeline Flow Visualization")
print("=" * 55)
if 'all_results' in dir() and all_results:
    # 첫 번째 이미지로 흐름도 생성 / 用第一张图生成流程图
    basename, res = all_results[0]
    out_dir = OUTPUT_DIR if 'OUTPUT_DIR' in dir() else "output"
    save_p = os.path.join(out_dir, f"{basename}_pipeline_flow.png")
    generate_pipeline_flowchart(res, image_name=basename, save_path=save_p)
    print(f"\nTo generate for all images / 모든 이미지에 대해 생성 / 为所有图像生成:")
    print("  for name, r in all_results:")
    print("      generate_pipeline_flowchart(r, image_name=name, save_path=f'output/{name}_pipeline_flow.png')")
else:
    print("No results found. Please run pipeline first (Cell 9).")
    print("all_results가 없습니다. 먼저 파이프라인을 실행하세요 (Cell 9).")
    print("未找到all_results。请先运行流水线（Cell 9）。")
